In [2]:

# SESSAO 09 - TAREFA 01 (VERSAO OTIMIZADA)

import pandas as pd
from pathlib import Path

pasta = Path("/content/drive/MyDrive/C1_SESSAO08_ATRIBUTOS")

# Ler apenas as colunas necessarias da matriz consolidada
matriz = pd.read_parquet(
    pasta / "08_Matriz_Treinamento_C1.parquet",
    columns=["Arquivo", "Poco", "Categoria"]
)

# Contar janelas produzidas por arquivo
contagem = matriz.groupby("Arquivo").size()

# Usar o manifesto V3 ja carregado
auditoria_09 = treino_v3[
    ["Arquivo", "Poco", "Categoria"]
].copy()

auditoria_09["Janelas"] = (
    auditoria_09["Arquivo"]
    .map(contagem)
    .fillna(0)
    .astype(int)
)

# Resumo por poco
resumo_09 = auditoria_09.groupby("Poco").agg(
    Arquivos=("Arquivo", "count"),
    Arquivos_com_janelas=("Janelas", lambda x: (x > 0).sum()),
    Janelas=("Janelas", "sum")
)

resumo_09["Arquivos_sem_janelas"] = (
    resumo_09["Arquivos"] - resumo_09["Arquivos_com_janelas"]
)

# Conferencias
assert len(auditoria_09) == 528
assert auditoria_09["Janelas"].sum() == 162326

print("TOTAL DE ARQUIVOS:", len(auditoria_09))
print("ARQUIVOS COM JANELAS:", (auditoria_09["Janelas"] > 0).sum())
print("ARQUIVOS SEM JANELAS:", (auditoria_09["Janelas"] == 0).sum())

print("\nPOCOS COM E SEM JANELAS:")
display(resumo_09.sort_values("Janelas"))

print("\nARQUIVOS SEM JANELAS POR CATEGORIA:")
display(
    auditoria_09.loc[
        auditoria_09["Janelas"] == 0, "Categoria"
    ].value_counts().sort_index().to_frame("Arquivos")
)

auditoria_09.to_csv(pasta / "09_Auditoria_Arquivos.csv", index=False)
resumo_09.to_csv(pasta / "09_Resumo_Pocos.csv")

print("\nTAREFA 01 EXECUTADA")


TOTAL DE ARQUIVOS: 528
ARQUIVOS COM JANELAS: 427
ARQUIVOS SEM JANELAS: 101

POCOS COM E SEM JANELAS:


,Arquivos,Arquivos_com_janelas,Janelas,Arquivos_sem_janelas
Poco,,,,
WELL-00007,12,0,0,12
WELL-00008,57,0,0,57
WELL-00028,1,0,0,1
WELL-00023,4,0,0,4
WELL-00026,3,0,0,3
WELL-00027,1,0,0,1
WELL-00032,1,0,0,1
WELL-00031,1,0,0,1
WELL-00035,6,0,0,6



ARQUIVOS SEM JANELAS POR CATEGORIA:


,Arquivos
Categoria,
0,59
4,10
7,4
8,7
9,21



TAREFA 01 EXECUTADA


In [3]:

# SESSAO 09 - TAREFA 02
# Diagnostico de um arquivo por poco sem janelas

import io
import time
import requests
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

sensores = ["P-TPT", "T-TPT", "P-ANULAR"]
tamanho, passo = 100, 50
codigos = list(range(10)) + list(range(101, 110))

# Selecionar um arquivo de cada poco sem janelas
amostra_09 = (
    auditoria_09[auditoria_09["Janelas"] == 0]
    .sort_values("Arquivo")
    .drop_duplicates("Poco")
)

resultados = []

with requests.Session() as sessao:

    for item in amostra_09.itertuples(index=False):

        url = (
            "https://raw.githubusercontent.com/petrobras/3W/"
            f"main/dataset/{item.Categoria}/{item.Arquivo}"
        )

        try:
            for tentativa in range(3):
                try:
                    resposta = sessao.get(url, timeout=60)
                    resposta.raise_for_status()
                    dados = pd.read_parquet(
                        io.BytesIO(resposta.content)
                    )
                    break
                except Exception:
                    if tentativa == 2:
                        raise
                    time.sleep(2 ** tentativa)

            rotulos = pd.to_numeric(
                dados["class"], errors="coerce"
            ).to_numpy(dtype=float, na_value=np.nan)

            sinais = dados.reindex(
                columns=sensores
            ).apply(
                pd.to_numeric, errors="coerce"
            ).to_numpy(dtype=float)

            inicios = np.arange(
                0, len(dados) - tamanho + 1, passo
            )

            # Janelas com rotulo original homogeneo
            r = sliding_window_view(
                rotulos, tamanho
            )[inicios]

            rotulo_ok = (
                np.isfinite(r).all(axis=1)
                & (r == r[:, :1]).all(axis=1)
                & np.isin(r[:, 0], codigos)
            )

            # Janelas temporalmente regulares
            if isinstance(dados.index, pd.DatetimeIndex):
                delta = np.diff(dados.index.asi8) / 1e9
                t = sliding_window_view(
                    delta, tamanho - 1
                )[inicios]
                tempo_ok = np.isclose(
                    t, 1, atol=0.1
                ).all(axis=1)
            else:
                tempo_ok = np.zeros(
                    len(inicios), dtype=bool
                )

            # Disponibilidade dos sensores
            s = sliding_window_view(
                np.isfinite(sinais), tamanho, axis=0
            )[inicios]

            sensor_ok = s.all(axis=2)

            resultados.append({
                "Poco": item.Poco,
                "Categoria": item.Categoria,
                "Arquivo": item.Arquivo,
                "Janelas_possiveis": len(inicios),
                "Rotulo_ok": int(rotulo_ok.sum()),
                "Tempo_ok": int(tempo_ok.sum()),
                "P-TPT_ok": int(sensor_ok[:, 0].sum()),
                "T-TPT_ok": int(sensor_ok[:, 1].sum()),
                "P-ANULAR_ok": int(sensor_ok[:, 2].sum()),
                "Tres_sensores_ok": int(
                    sensor_ok.all(axis=1).sum()
                ),
                "Janelas_finais": int(
                    (
                        rotulo_ok
                        & tempo_ok
                        & sensor_ok.all(axis=1)
                    ).sum()
                )
            })

        except Exception as erro:
            resultados.append({
                "Poco": item.Poco,
                "Arquivo": item.Arquivo,
                "Erro": str(erro)
            })

diagnostico_09 = pd.DataFrame(resultados)

print("DIAGNOSTICO DOS 13 POCOS:")
display(diagnostico_09)

diagnostico_09.to_csv(
    pasta / "09_Diagnostico_Inicial_Descartes.csv",
    index=False
)

print("\nTAREFA 02 - DIAGNOSTICO INICIAL EXECUTADO")



DIAGNOSTICO DOS 13 POCOS:


,Poco,Categoria,Arquivo,Janelas_possiveis,Rotulo_ok,Tempo_ok,P-TPT_ok,T-TPT_ok,P-ANULAR_ok,Tres_sensores_ok,Janelas_finais
0,WELL-00007,4,WELL-00007_20170517200012.parquet,214,142,214,214,214,0,0,0
1,WELL-00008,0,WELL-00008_20170610210246.parquet,423,351,423,0,0,0,0,0
2,WELL-00023,7,WELL-00023_20180826212652.parquet,11198,11124,11198,0,0,0,0,0
3,WELL-00026,8,WELL-00026_20160601130000.parquet,3743,3668,3743,0,0,3743,0,0
4,WELL-00027,8,WELL-00027_20230918170106.parquet,13607,13531,13607,13607,13607,0,0,0
5,WELL-00028,8,WELL-00028_20210617053128.parquet,6506,6430,6506,6492,0,6492,0,0
6,WELL-00031,8,WELL-00031_20140403170404.parquet,7249,7173,7249,7247,0,0,0,0
7,WELL-00032,8,WELL-00032_20110830235321.parquet,2683,2607,2683,2683,2683,0,0,0
8,WELL-00034,9,WELL-00034_20190825004401.parquet,377,305,377,377,0,0,0,0
9,WELL-00035,9,WELL-00035_20190819093245.parquet,1072,1000,1072,1072,0,0,0,0



TAREFA 02 - DIAGNOSTICO INICIAL EXECUTADO


In [4]:

# SESSAO 09 - TAREFA 03
# Verificar sensores alternativos em dois pocos

import io
import requests
import pandas as pd

pocos_investigar = ["WELL-00008", "WELL-00023"]

amostra = amostra_09[
    amostra_09["Poco"].isin(pocos_investigar)
]

resultados = []

for item in amostra.itertuples(index=False):

    url = (
        "https://raw.githubusercontent.com/petrobras/3W/"
        f"main/dataset/{item.Categoria}/{item.Arquivo}"
    )

    resposta = requests.get(url, timeout=90)
    resposta.raise_for_status()

    dados = pd.read_parquet(
        io.BytesIO(resposta.content)
    )

    # Considerar apenas registros com rotulos conhecidos
    codigos = list(range(10)) + list(range(101, 110))
    mask = dados["class"].isin(codigos)

    for sensor in dados.columns:

        # Rótulos não são atributos preditores
        if sensor in ["class", "state"]:
            continue

        valores = pd.to_numeric(
            dados.loc[mask, sensor],
            errors="coerce"
        )

        validos = valores.notna().sum()

        resultados.append({
            "Poco": item.Poco,
            "Sensor": sensor,
            "Validos": validos,
            "Disponibilidade_%": round(
                100 * validos / max(mask.sum(), 1), 2
            ),
            "Valores_distintos": valores.nunique(
                dropna=True
            )
        })

diagnostico_sensores = pd.DataFrame(resultados)

for poco in pocos_investigar:

    print("\nPOCO:", poco)

    tabela = diagnostico_sensores[
        diagnostico_sensores["Poco"] == poco
    ].sort_values(
        ["Disponibilidade_%", "Valores_distintos"],
        ascending=False
    )

    display(tabela.head(15))

diagnostico_sensores.to_csv(
    pasta / "09_Sensores_Alternativos.csv",
    index=False
)

print("\nTAREFA 03 - AUDITORIA INICIAL EXECUTADA")



POCO: WELL-00008


,Poco,Sensor,Validos,Disponibilidade_%,Valores_distintos
13,WELL-00008,P-JUS-CKGL,17609,100.0,2607
0,WELL-00008,ABER-CKGL,0,0.0,0
1,WELL-00008,ABER-CKP,0,0.0,0
2,WELL-00008,ESTADO-DHSV,0,0.0,0
3,WELL-00008,ESTADO-M1,0,0.0,0
4,WELL-00008,ESTADO-M2,0,0.0,0
5,WELL-00008,ESTADO-PXO,0,0.0,0
6,WELL-00008,ESTADO-SDV-GL,0,0.0,0
7,WELL-00008,ESTADO-SDV-P,0,0.0,0
8,WELL-00008,ESTADO-W1,0,0.0,0



POCO: WELL-00023


,Poco,Sensor,Validos,Disponibilidade_%,Valores_distintos
40,WELL-00023,P-JUS-CKGL,556389,100.0,103156
43,WELL-00023,P-MON-CKP,556389,100.0,7016
34,WELL-00023,ESTADO-SDV-P,556389,100.0,1
27,WELL-00023,ABER-CKGL,0,0.0,0
28,WELL-00023,ABER-CKP,0,0.0,0
29,WELL-00023,ESTADO-DHSV,0,0.0,0
30,WELL-00023,ESTADO-M1,0,0.0,0
31,WELL-00023,ESTADO-M2,0,0.0,0
32,WELL-00023,ESTADO-PXO,0,0.0,0
33,WELL-00023,ESTADO-SDV-GL,0,0.0,0



TAREFA 03 - AUDITORIA INICIAL EXECUTADA


In [5]:

# SESSAO 09 - TAREFA 03
# Cobertura inicial de sensores nos 28 pocos

import io
import requests
import pandas as pd
from tqdm.auto import tqdm

sensores_candidatos = [
    "P-TPT", "T-TPT", "P-ANULAR",
    "P-JUS-CKGL", "P-MON-CKP"
]

# Selecionar um arquivo por poco da V3
representantes = (
    treino_v3.sort_values(["Poco", "Arquivo"])
    .drop_duplicates("Poco")
)

resultados = []
falhas = []

with requests.Session() as sessao:

    for item in tqdm(
        representantes.itertuples(index=False),
        total=len(representantes)
    ):

        url = (
            "https://raw.githubusercontent.com/petrobras/3W/"
            f"main/dataset/{item.Categoria}/{item.Arquivo}"
        )

        try:
            resposta = sessao.get(url, timeout=90)
            resposta.raise_for_status()

            dados = pd.read_parquet(
                io.BytesIO(resposta.content)
            )

            rotulos = pd.to_numeric(
                dados["class"], errors="coerce"
            )

            validos = rotulos.isin(
                list(range(10))
                + list(range(101, 110))
            )

            for sensor in sensores_candidatos:

                serie = pd.to_numeric(
                    dados.loc[validos, sensor],
                    errors="coerce"
                )

                percentual = (
                    100 * serie.notna().mean()
                    if len(serie) else 0
                )

                resultados.append({
                    "Poco": item.Poco,
                    "Sensor": sensor,
                    "Disponibilidade_%": round(
                        percentual, 2
                    )
                })

        except Exception as erro:
            falhas.append({
                "Poco": item.Poco,
                "Erro": str(erro)
            })

cobertura = pd.DataFrame(resultados)

tabela_cobertura = cobertura.pivot(
    index="Poco",
    columns="Sensor",
    values="Disponibilidade_%"
)

print("COBERTURA DOS SENSORES POR POCO (%):")
display(tabela_cobertura)

print("\nPOCOS COM MAIS DE 90% DE DADOS:")
display(
    (tabela_cobertura > 90)
    .sum()
    .sort_values(ascending=False)
    .to_frame("Pocos")
)

print("\nPOCOS ANALISADOS:",
      tabela_cobertura.shape[0])
print("FALHAS:", len(falhas))

if falhas:
    display(pd.DataFrame(falhas))

tabela_cobertura.to_csv(
    pasta / "09_Cobertura_Sensores_Pocos.csv"
)

print("\nTAREFA 03 - COMPARACAO EXECUTADA")


  0%|          | 0/28 [00:00<?, ?it/s]

COBERTURA DOS SENSORES POR POCO (%):


Sensor,P-ANULAR,P-JUS-CKGL,P-MON-CKP,P-TPT,T-TPT
Poco,,,,,
WELL-00002,100.00,0.00,0.00,100.00,100.00
WELL-00007,0.00,100.00,100.00,100.00,100.00
WELL-00008,0.00,100.00,0.00,0.00,0.00
WELL-00009,100.00,100.00,100.00,100.00,100.00
WELL-00011,92.66,0.00,0.00,92.66,92.66
WELL-00012,100.00,0.00,100.00,100.00,100.00
WELL-00013,100.00,0.00,0.00,100.00,100.00
WELL-00014,100.00,100.00,100.00,100.00,100.00
WELL-00020,100.00,100.00,100.00,100.00,100.00



POCOS COM MAIS DE 90% DE DADOS:


,Pocos
Sensor,
P-TPT,25
P-MON-CKP,24
P-JUS-CKGL,23
T-TPT,20
P-ANULAR,17



POCOS ANALISADOS: 28
FALHAS: 0

TAREFA 03 - COMPARACAO EXECUTADA


In [7]:

# SESSAO 09 - TAREFA 04 (VERSAO CORRIGIDA)
# Cobertura das categorias e rotulos por poco

import pandas as pd

# 1. Recuperar a matriz salva na Sessao 08
matriz_treino = pd.read_parquet(
    pasta / "08_Matriz_Treinamento_C1.parquet",
    columns=["Arquivo", "Poco", "Categoria", "Rotulo_C1"]
)

print("JANELAS RECUPERADAS:", len(matriz_treino))

# 2. Cobertura original por categoria de arquivo
original = treino_v3.groupby("Categoria").agg(
    Arquivos_V3=("Arquivo", "nunique"),
    Pocos_V3=("Poco", "nunique")
)

preparado = matriz_treino.groupby("Categoria").agg(
    Arquivos_com_janelas=("Arquivo", "nunique"),
    Pocos_com_janelas=("Poco", "nunique")
)

cobertura_arquivos = original.join(
    preparado, how="left"
).fillna(0).astype(int)

print("\nCOBERTURA POR CATEGORIA DE ARQUIVO:")
display(cobertura_arquivos)

# 3. Cobertura por rotulo efetivo das janelas
cobertura_rotulos = matriz_treino.groupby(
    "Rotulo_C1"
).agg(
    Janelas=("Arquivo", "size"),
    Arquivos=("Arquivo", "nunique"),
    Pocos=("Poco", "nunique")
).reindex(range(10), fill_value=0)

cobertura_rotulos.index.name = "Classe"

cobertura_rotulos["Situacao"] = (
    cobertura_rotulos["Pocos"].apply(
        lambda n: (
            "Somente 1 poco" if n == 1
            else "Mais de 1 poco" if n > 1
            else "Sem janelas"
        )
    )
)

print("\nCOBERTURA POR ROTULO C1:")
display(cobertura_rotulos)

print("\nCLASSES COM SOMENTE UM POCO:")
print(
    cobertura_rotulos.index[
        cobertura_rotulos["Pocos"] == 1
    ].tolist()
)

print("\nCLASSES COM MAIS DE UM POCO:")
print(
    cobertura_rotulos.index[
        cobertura_rotulos["Pocos"] > 1
    ].tolist()
)

# 4. Salvar no Drive
cobertura_arquivos.to_csv(
    pasta / "09_Cobertura_Categorias_Arquivos.csv"
)

cobertura_rotulos.to_csv(
    pasta / "09_Cobertura_Rotulos_C1.csv"
)

print("\nTAREFA 04 EXECUTADA")


JANELAS RECUPERADAS: 162326

COBERTURA POR CATEGORIA DE ARQUIVO:


,Arquivos_V3,Pocos_V3,Arquivos_com_janelas,Pocos_com_janelas
Categoria,,,,
0.0,268,3,209,1
1.0,1,1,1,1
2.0,18,5,18,5
3.0,31,1,31,1
4.0,143,3,133,2
5.0,6,1,6,1
6.0,3,1,3,1
7.0,6,2,2,1
8.0,10,8,3,3



COBERTURA POR ROTULO C1:


,Janelas,Arquivos,Pocos,Situacao
Classe,,,,
0,102708,263,15,Mais de 1 poco
1,188,1,1,Somente 1 poco
2,1700,18,5,Mais de 1 poco
3,10956,31,1,Somente 1 poco
4,18793,133,2,Mais de 1 poco
5,6104,6,1,Somente 1 poco
6,335,3,1,Somente 1 poco
7,864,2,1,Somente 1 poco
8,16904,3,3,Mais de 1 poco



CLASSES COM SOMENTE UM POCO:
[1, 3, 5, 6, 7]

CLASSES COM MAIS DE UM POCO:
[0, 2, 4, 8, 9]

TAREFA 04 EXECUTADA


In [8]:

# SESSAO 09 - TAREFA 05
# Random Forest com validacao cruzada por poco
# Experimento diagnostico - matriz original da Sessao 08

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# 1. Recuperar a matriz completa
dados = pd.read_parquet(
    pasta / "08_Matriz_Treinamento_C1.parquet"
)

metadados = [
    "Arquivo", "Poco", "Categoria", "Inicio", "Fim",
    "Rotulo_original", "Rotulo_C1", "Fase"
]

atributos = [
    c for c in dados.columns if c not in metadados
]

assert len(atributos) == 27

# 2. Limitar a influencia dos arquivos/pocos longos
# Maximo de 500 janelas por poco e classe
partes = []

for _, grupo in dados.groupby(["Poco", "Rotulo_C1"]):
    partes.append(
        grupo.sample(
            n=min(len(grupo), 500),
            random_state=42
        )
    )

amostra_rf = pd.concat(
    partes, ignore_index=True
)

print("JANELAS ORIGINAIS:", len(dados))
print("JANELAS NA AMOSTRA:", len(amostra_rf))
print("POCOS REPRESENTADOS:", amostra_rf["Poco"].nunique())

# 3. Entradas e rotulos
X = amostra_rf[atributos]
y = amostra_rf["Rotulo_C1"].astype(int)
pocos = amostra_rf["Poco"]

# Classes com exemplos de pelo menos dois pocos
pocos_por_classe = (
    amostra_rf.groupby("Rotulo_C1")["Poco"].nunique()
)

classes_multi_poco = sorted(
    pocos_por_classe[pocos_por_classe >= 2].index.tolist()
)

print("CLASSES COM MAIS DE UM POCO:", classes_multi_poco)

# 4. Validacao cruzada agrupada
gkf = GroupKFold(n_splits=3)

resultados_folds = []
predicoes_folds = []

for fold, (idx_t, idx_v) in enumerate(
    gkf.split(X, y, groups=pocos),
    start=1
):

    X_t, X_v = X.iloc[idx_t], X.iloc[idx_v]
    y_t, y_v = y.iloc[idx_t], y.iloc[idx_v]

    assert set(pocos.iloc[idx_t]).isdisjoint(
        set(pocos.iloc[idx_v])
    )

    modelo = Pipeline([
        ("imputacao", SimpleImputer(
            strategy="median"
        )),
        ("rf", RandomForestClassifier(
            n_estimators=80,
            max_depth=12,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        ))
    ])

    modelo.fit(X_t, y_t)
    previsto = modelo.predict(X_v)

    presentes = sorted(y_v.unique())

    f1 = f1_score(
        y_v, previsto,
        labels=presentes,
        average="macro",
        zero_division=0
    )

    resultados_folds.append({
        "Fold": fold,
        "Pocos_treino": pocos.iloc[idx_t].nunique(),
        "Pocos_validacao": pocos.iloc[idx_v].nunique(),
        "Janelas_validacao": len(idx_v),
        "Classes_treino": str(sorted(y_t.unique())),
        "Classes_validacao": str(presentes),
        "F1_macro_presentes": round(f1, 4)
    })

    registro = amostra_rf.iloc[idx_v][
        ["Arquivo", "Poco", "Rotulo_C1", "Fase"]
    ].copy()

    registro["Fold"] = fold
    registro["Predicao_RF"] = previsto

    predicoes_folds.append(registro)

# 5. Reunir os resultados de todos os folds
resumo_folds = pd.DataFrame(resultados_folds)
predicoes_cv = pd.concat(
    predicoes_folds, ignore_index=True
)

print("\nRESULTADOS POR FOLD:")
display(resumo_folds)

# 6. Avaliacao conjunta das predicoes fora do treino
y_real = predicoes_cv["Rotulo_C1"].astype(int)
y_previsto = predicoes_cv["Predicao_RF"].astype(int)

f1_multi = f1_score(
    y_real, y_previsto,
    labels=classes_multi_poco,
    average="macro",
    zero_division=0
)

print("\nF1 MACRO - CLASSES COM MULTIPLOS POCOS:")
print(round(f1_multi, 4))
print("Classes consideradas:", classes_multi_poco)

relatorio = classification_report(
    y_real, y_previsto,
    labels=list(range(10)),
    output_dict=True,
    zero_division=0
)

metricas_cv = pd.DataFrame(relatorio).T

print("\nMETRICAS POR CLASSE:")
display(
    metricas_cv.loc[
        [str(i) for i in range(10)],
        ["precision", "recall", "f1-score", "support"]
    ].round(3)
)

# 7. Salvar no Drive
resumo_folds.to_csv(
    pasta / "09_Resultados_Folds_RF.csv",
    index=False
)

predicoes_cv.to_parquet(
    pasta / "09_Predicoes_CV_RF.parquet",
    index=False
)

metricas_cv.to_csv(
    pasta / "09_Metricas_CV_RF.csv"
)

print("\nTAREFA 05 EXECUTADA")


JANELAS ORIGINAIS: 162326
JANELAS NA AMOSTRA: 13071
POCOS REPRESENTADOS: 15
CLASSES COM MAIS DE UM POCO: [0, 2, 4, 8, 9]

RESULTADOS POR FOLD:


,Fold,Pocos_treino,Pocos_validacao,Janelas_validacao,Classes_treino,Classes_validacao,F1_macro_presentes
0,1,11,4,4269,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[np.int64(0), np.int64(2), np.int64(3), np.int...",0.1766
1,2,10,5,4368,"[np.int64(0), np.int64(2), np.int64(3), np.int...","[np.int64(0), np.int64(1), np.int64(2), np.int...",0.2373
2,3,9,6,4434,"[np.int64(0), np.int64(1), np.int64(2), np.int...","[np.int64(0), np.int64(2), np.int64(5), np.int...",0.0512



F1 MACRO - CLASSES COM MULTIPLOS POCOS:
0.2226
Classes consideradas: [0, 2, 4, 8, 9]

METRICAS POR CLASSE:


,precision,recall,f1-score,support
0,0.435,0.472,0.452,6212.0
1,0.000,0.000,0.000,188.0
2,0.160,0.558,0.248,843.0
3,0.000,0.000,0.000,500.0
4,0.000,0.000,0.000,1000.0
5,0.000,0.000,0.000,500.0
6,0.000,0.000,0.000,335.0
7,0.000,0.000,0.000,500.0
8,0.373,0.331,0.351,1500.0
9,0.227,0.035,0.061,1493.0



TAREFA 05 EXECUTADA
